# Acquire a source file (F04.2)

Validate a staged source file and land it immutably under the managed
source Volume. This notebook is acquisition-only: it never parses
Airbnb records and never writes Bronze or Silver tables.

## Manual upload route (no outbound network needed)

1. Download the source file on any machine with internet access.
2. In Databricks, open Catalog Explorer → `trippilot_dev` → `bronze` →
   `Volumes` → `source_files`.
3. Open `_incoming` (create `_incoming/<source_family>/<entity>`
   subfolders as needed) and use **Upload to this volume**.
4. Run this notebook with the staged path in the widget below.

Landing is idempotent: re-running with the same bytes changes nothing. A
corrected file (different SHA-256) lands as a separate hash-versioned
object; nothing is ever overwritten.

In [0]:
# Source identity and provenance inputs (F04.2-T01).
dbutils.widgets.text(
    "source_path",
    "/Volumes/trippilot_dev/bronze/source_files/_incoming/insideairbnb/toronto/listings.csv.gz",
    "Staged source path (under _incoming)",
)
dbutils.widgets.text("source_family", "insideairbnb", "Source family")
dbutils.widgets.text("source_entity_id", "toronto", "Source entity ID (city, agency, document)")
dbutils.widgets.text("source_version", "2026-09-16", "Source version or snapshot")
dbutils.widgets.text("file_type", "csv.gz", "File type")
dbutils.widgets.text("source_url", "https://insideairbnb.com/get-the-data/", "Official source URL")
dbutils.widgets.text("schema_version", "pending", "Schema version")
dbutils.widgets.text("license", "CC0-1.0", "Licence")
dbutils.widgets.text(
    "attribution", "Inside Airbnb (insideairbnb.com), CC0 1.0", "Attribution"
)
dbutils.widgets.text(
    "retrieved_at_utc", "", "Retrieved at (UTC, e.g. 2026-11-13T18:00:00Z; blank = now)"
)

In [0]:
# Locate the repository root and import the shared acquisition module.
from datetime import UTC, datetime
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in [start, *start.parents]
        if (p / "src").is_dir() and (p / "config/project.yml").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Cannot locate repository root. Open this notebook inside the Databricks Git folder."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository root:", ROOT)

from src.common.source_acquisition import acquire_source_file
from src.common.source_manifest import compute_file_hash_sha256

SOURCE_ROOT = "/Volumes/trippilot_dev/bronze/source_files"

In [0]:
# Build the manifest metadata from the widgets and land the file.
source_path = dbutils.widgets.get("source_path")

retrieved_at = dbutils.widgets.get("retrieved_at_utc")
if not retrieved_at:
    retrieved_at = datetime.now(UTC).isoformat().replace("+00:00", "Z")

source_metadata = {
    "source_family": dbutils.widgets.get("source_family"),
    "source_entity_id": dbutils.widgets.get("source_entity_id"),
    "source_version": dbutils.widgets.get("source_version"),
    "file_type": dbutils.widgets.get("file_type"),
    "source_url": dbutils.widgets.get("source_url"),
    "retrieved_at_utc": retrieved_at,
    "schema_version": dbutils.widgets.get("schema_version"),
    "license": dbutils.widgets.get("license"),
    "attribution": dbutils.widgets.get("attribution"),
    "source_path": source_path,
}

record = acquire_source_file(source_metadata, source_path, SOURCE_ROOT)

In [0]:
# Result: the landed manifest record (F04.1 contract) with hash verification.
landed_hash = compute_file_hash_sha256(record["landing_path"])
assert landed_hash == record["file_hash_sha256"], "Landed file hash does not match the record"

print(f"source_file_id   : {record['source_file_id']}")
print(f"processing_state : {record['processing_state']}")
print(f"landing_path     : {record['landing_path']}")
print(f"file_hash_sha256 : {record['file_hash_sha256']}")
print(f"byte_size        : {record['byte_size']}")
print(f"run_id           : {record['run_id']}")
print("Landed-file hash verified: ", landed_hash)
print("\nThe manifest record is returned here for inspection. Durable Delta")
print("registration of manifest records arrives with F04.3 (manifest store).")